## 🧪 Experiments


In [ ]:
# Imports
from dotenv         import load_dotenv
import pandas       as pd
import datetime
import traceback
import os

# Custom Imports
import sys
sys.path.append('../')
import AppUtils
import LLMUtils
from AnalysisUtils import AnalysisUtils, LlmRetryLimitExceededError, ObfuscationClassificationAnalysisUtils

##### Initialization


In [ ]:
print("⚡ START: {} ⚡".format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
initTime = datetime.datetime.now()

In [ ]:
# Paths
TMP_PATH = "../../0_Data/TMP"

if not os.path.exists(TMP_PATH):
    os.makedirs(TMP_PATH)
    print("--- 🆕 Folder created : {}".format(TMP_PATH))
else:
    print("--- 📂 TMP Folder ready : {}".format(TMP_PATH))

In [ ]:
# GPU Test
import torch

# Check if CUDA is available
print("--- 🖥️ CUDA available           : {}".format(torch.cuda.is_available()))

# Show the number of GPUs available
numGpus = torch.cuda.device_count()
print("--- 🖥️ Number of available GPUs : {}".format(numGpus))

# Display the names of the GPUs
for i in range(numGpus):
	print("--- 🖥️ GPU {}: {}".format(i, torch.cuda.get_device_name(i)))

In [ ]:
# Load .env Info
load_dotenv()

#### 📥 1] Load Data


In [ ]:
# Where the input APKs are located
INPUT_APKS_PATH = os.path.abspath(os.path.join(os.getcwd(), "../../0_Data/InputApks/ClassRename"))

# Get the obfuscation technique from the folder name
obfuscationTechniqueDirName = os.path.basename(os.path.normpath(INPUT_APKS_PATH))
obfuscationTechniqueField   = obfuscationTechniqueDirName

# Load the input APKs information into a DataFrame
apkRecords = []
print("\n--- ⭕ Loading APKs...")
for apkFileName in sorted(os.listdir(INPUT_APKS_PATH)):
	if not apkFileName.endswith(".apk"):
		continue

	alreadyDownloadedPath = os.path.join(INPUT_APKS_PATH, apkFileName)
	sha256                = AppUtils.App.computeFileSha256(alreadyDownloadedPath)
	apkStem               = os.path.splitext(apkFileName)[0]
	if apkStem.endswith("_obfuscated"):
		apkStem = apkStem[:-len("_obfuscated")]
	pkgName = apkStem.rsplit("_", 1)[0]

	apkRecords.append({
		"sha256"                 : sha256,
		"pkgName"                : pkgName,
		"obfuscationTechnique"  : obfuscationTechniqueField,
		"alreadyDownloadedPath"  : alreadyDownloadedPath
	})

appsDF = pd.DataFrame(apkRecords)
print("--- 🔹 Input APKs Path       : {}".format(INPUT_APKS_PATH))
print("--- 🔹 Obfuscation Technique : {}".format(obfuscationTechniqueField))
print("--- #️⃣ Number of Apps        : {}".format(len(appsDF)))

if len(appsDF) == 0:
	raise ValueError("No APK files found in {}".format(INPUT_APKS_PATH))

# Test purposes
appsDF = appsDF.head(3)

# Show first 3 rows
appsDF.head(3)

### 🧪 2] LLM Analysis


In [ ]:
# PAYING
# MODEL = "gpt-4o-mini"

# PAYING
# MODEL = "gemini-3-flash-preview"

# FREE
# MODEL = "gpt-oss:20b"
# MODEL = "deepseek-r1:32b"
# MODEL = "gemma3:27b"
MODEL = "qwen3:30b"
# MODEL = "llama3.1:8b"
# MODEL = "phi3:14b"

# Context Window Size
CONTEXT_WINDOW_SIZE = 128000
CONTEXT_THRESHOLD   = 0.8 * CONTEXT_WINDOW_SIZE

# Check and Init
print("\n--- ⭕ LLM Init & Check ...")
print("--- 🔸 Model: {}".format(MODEL))

# PAYING
if LLMUtils.isOpenAiModel(MODEL):
	llmInterface = LLMUtils.OpenAiInterface(model=MODEL,contextWindow=CONTEXT_WINDOW_SIZE)
elif MODEL.lower().startswith("gemini"):
	llmInterface = LLMUtils.GeminiInterface(model=MODEL,contextWindow=CONTEXT_WINDOW_SIZE)
else:
	llmInterface = LLMUtils.OllamaInterface(model=MODEL,contextWindow=CONTEXT_WINDOW_SIZE)

promptTokenizer = LLMUtils.OpenAiTokenizer()

# Send a test request to check if the LLM is working
print("--- 🔸 LLM Response : {}".format(llmInterface.sendRequest("Ping!")))

##### Parameters


In [ ]:
# To compute the minimum number of classes for statistically significant random sample
CONFIDENCE_LEVEL    = 95
ERROR_MARGIN        = 5

# Obfuscation techniques i.e. potential labels
OBFUSCATION_TECHNIQUES = [
    "ArithmeticBranch",
    "CallIndirection",
    "ClassRename",
    "ConstStringEncryption",
    "FieldRename",
    "Goto",
    "MethodOverload",
    "MethodRename",
    "Reflection",
    "ResStringEncryption",
]

# Labels used by the classifier. Keep the full list for multi-class classification.
# For single-technique debugging, you can temporarily set EXPECTED_LABELS = ["ClassRename"].
EXPECTED_LABELS = OBFUSCATION_TECHNIQUES

# Filtering [None | "system" | "tp" | "both" | "pkgNameOnly"]
FILTERING = "both"

# Sampling
RANDOM_SEED = 777

# LLM Robustness
NUM_ITERATIONS  = 3
MAX_RETRIES     = 3

# Logging
SILENT_MODE = True

In [ ]:
# Prompt Selection
PROMPTS_PATH = "../prompt.yaml"
PROMPT_ID    = "ObfuscationClassificationV1"

prompts         = AnalysisUtils.loadPrompts(PROMPTS_PATH)
promptInfo      = AnalysisUtils.getPromptById(prompts, PROMPT_ID)
promptTemplate  = promptInfo["promptTemplate"]

print("--- 📝 Prompt ID          : {}".format(promptInfo["promptID"]))
print("--- 📝 Prompt Description : {}".format(promptInfo["promptDescription"]))
print("--- 📝 Prompt Template    : {}".format(promptTemplate))

In [ ]:
# Results & Resume
RESULTS_DIR_PATH    = os.path.abspath(os.path.join(os.getcwd(), "../../0_Data/Results/", PROMPT_ID, obfuscationTechniqueDirName))
MODEL_FILE_NAME     = MODEL.replace(":", "_").replace("/", "_")
OUTPUT_PATH_JSON    = os.path.join(RESULTS_DIR_PATH, "results_{}.json".format(MODEL_FILE_NAME))
OUTPUT_PATH_CSV     = os.path.join(RESULTS_DIR_PATH, "results_{}.csv".format(MODEL_FILE_NAME))

# Create results directory if it doesn't exist
classificationResults   = AnalysisUtils.loadExistingResults(OUTPUT_PATH_JSON)
completedSha256Set      = {result["sha256"] for result in classificationResults}
pendingAppsDF           = appsDF[~appsDF["sha256"].isin(completedSha256Set)].reset_index(drop=True)

print("\n--- ⭕ Checking for existing results...")
if len(classificationResults) > 0:
    print("--- ☑️ File Found!")
    print("--- 🔄️ Current Progress : {}/{}".format(len(classificationResults), len(appsDF)))
else:
    print("--- 🆕 No existing results file found! --> Starting fresh analysis...")


In [ ]:
# Basic App Analysis
for appIdx, appRow in pendingAppsDF.iterrows():
    # Extract app information from the DataFrame row for easier access and readability during processing.
    sha256                  = appRow["sha256"]
    pkgName                 = appRow["pkgName"]
    obfuscationTechnique    = appRow["obfuscationTechnique"]
    alreadyDownloadedPath   = appRow["alreadyDownloadedPath"]

    # Initialize variables for the app and its analysis result, which will be populated during processing. This allows us to handle cleanup and result saving in the finally block, even if errors occur.
    app         = None
    appResult   = None

    print("--- 🔑 Analyzing App [{}/{}] : {}".format(appIdx + 1, len(pendingAppsDF), sha256))
    print("--- 📦 pkgName               : {}".format(pkgName))

    try:
        app = AppUtils.App(sha256, pkgName, TMP_PATH + "/", downloadedApkPath=alreadyDownloadedPath)
        app.decompileWithApktool()
        manifestPkgName = app.getPkgNameFromManifest()
        if manifestPkgName is not None and manifestPkgName != app.pkgName:
            print("--- 🔄 pkgName refreshed from manifest: {} -> {}".format(app.pkgName, manifestPkgName))
            app.pkgName = manifestPkgName
            pkgName     = manifestPkgName
        app.collectSmaliClasses()

        print("\n--- ⭕ Filtering Smali Classes before sampling")
        print("--- 🔹 Filtering Strategy : {}".format(FILTERING))
        if FILTERING is None:
            print("--- 🔹 No filtering applied.")
        elif FILTERING == "system":
            app.filterOutSystemLibraries()
            app.filterOutClassesContainingDollarSign()
        elif FILTERING == "tp":
            app.filterOutThirdPartyLibraries()
            app.filterOutClassesContainingDollarSign()
        elif FILTERING == "both":
            app.filterOutSystemLibraries()
            app.filterOutThirdPartyLibraries()
            app.filterOutClassesContainingDollarSign()
        elif FILTERING == "pkgNameOnly":
            app.filterByPkgName()
            app.filterOutClassesContainingDollarSign()
        else:
            raise ValueError("Unsupported FILTERING mode: {}".format(FILTERING))

        if app.numSmaliClasses == 0:
            print("--- ⚠️ No Smali Classes found.")
            appResult = AnalysisUtils.createResultsObject(
                sha256                  = sha256,
                pkgName                 = pkgName,
                obfuscationTechnique    = obfuscationTechnique,
                status                  = "NO_SMALI_CLASSES",
                numSmaliClasses         = app.numSmaliClasses,
                numSmaliClassesAnalyzed = 0,
                llmFinalLabel           = None
            )
            continue

        print("\n--- ⭕ Computing random sample size [confidence={}%, error margin={}%]".format(CONFIDENCE_LEVEL, ERROR_MARGIN))
        numSmaliClassesAnalyzed = AnalysisUtils.computeRandomSampleSize(app.numSmaliClasses, CONFIDENCE_LEVEL, ERROR_MARGIN)
        print("--- #️⃣ Random Sample Size : {}".format(numSmaliClassesAnalyzed))

        # Test purposes
        numSmaliClassesAnalyzed = 3

        # Sample the smali classes to analyze, ensuring reproducibility with a fixed random seed. Initialize counters for effective number of classes analyzed (after filtering out those that exceed context threshold) and number of classes skipped due to context threshold.
        sampledSmaliClasses                 = AnalysisUtils.getRandomSample(app.smaliClasses, numSmaliClassesAnalyzed, RANDOM_SEED)
        effectiveNumSmaliClassesAnalyzed    = numSmaliClassesAnalyzed
        numSkippedForContextThreshold       = 0
        numSkippedForInvalidLlmOutput       = 0
        appLabelCounts = {}

        print("\n--- ⭕ Analyzing Smali Classes with LLM")
        print("--- 🔹 Num Iterations        : {}".format(NUM_ITERATIONS))
        print("--- 🔹 Max Retries           : {}\n".format(MAX_RETRIES))

        # For each sampled smali class, build the prompt and check its token size against the context threshold. If it exceeds the threshold, skip this class and update the count of skipped classes. Otherwise, analyze the class with the LLM and update the label counts for majority voting.
        for smaliIdx, smaliClass in enumerate(sampledSmaliClasses):
            if not SILENT_MODE:
                print("--- 🔸 Checking Smali Class [{}/{}]: {}".format(smaliIdx + 1, numSmaliClassesAnalyzed, smaliClass["className"]))

            # Check prompt size against context threshold before analyzing. If it exceeds the threshold, skip this class and update the count of skipped classes. Otherwise, analyze the class with the LLM and update the label counts for majority voting.
            try:
                prompt          = ObfuscationClassificationAnalysisUtils.buildClassificationPrompt(promptTemplate, smaliClass, EXPECTED_LABELS)
                promptNumTokens = promptTokenizer.getNumTokens(prompt)
            except Exception as classStepExc:
                print("--- ⚠️ Prompt/tokenization failure for class: {}".format(smaliClass["className"]))
                print("--- ⚠️ Expected labels: {}".format(EXPECTED_LABELS))
                raise
            if promptNumTokens > CONTEXT_THRESHOLD:
                effectiveNumSmaliClassesAnalyzed    -= 1
                numSkippedForContextThreshold       += 1
                print("--- ⏭️ Skipping Smali Class [{}/{}] : {} | Prompt too large ({} tokens > {}).".format(smaliIdx + 1, numSmaliClassesAnalyzed, smaliClass["className"], promptNumTokens, CONTEXT_THRESHOLD))
                if not SILENT_MODE:
                    print("---" * 20)
                continue

            # Analyze the smali class with the LLM and update the label counts for majority voting.
            try:
                classAnalysis   = ObfuscationClassificationAnalysisUtils.analyzeSmaliClassWithMajorityVote(llmInterface, smaliClass, promptTemplate, EXPECTED_LABELS, NUM_ITERATIONS, MAX_RETRIES)
                classLabel      = classAnalysis["majorityLabel"]
            except LlmRetryLimitExceededError as exc:
                effectiveNumSmaliClassesAnalyzed -= 1
                numSkippedForInvalidLlmOutput += 1
                print("--- ⏭️ Skipping Smali Class [{}/{}] : {} | Invalid LLM output after {} retries: {}".format(smaliIdx + 1, numSmaliClassesAnalyzed, smaliClass["className"], MAX_RETRIES, exc))
                if not SILENT_MODE:
                    print("---" * 20)
                continue
            except Exception:
                print("--- ⚠️ LLM classification failure for class: {}".format(smaliClass["className"]))
                print("--- ⚠️ Expected labels: {}".format(EXPECTED_LABELS))
                raise
            appLabelCounts[classLabel] = appLabelCounts.get(classLabel, 0) + 1

            # Log the analysis details for this class
            if not SILENT_MODE:
                print("--- 🏷️ Label Counts   : {}".format(classAnalysis["labelCounts"]))
                print("--- 🏷️ Majority Label : {}".format(classLabel))
                print("---" * 20)

        if effectiveNumSmaliClassesAnalyzed == 0:
            print("\n--- ⚠️ No sampled Smali Classes could be analyzed.")
            print("--- 🔹 N. Skipped for Context      : {}".format(numSkippedForContextThreshold))
            print("--- 🔹 N. Skipped for LLM Output   : {}".format(numSkippedForInvalidLlmOutput))
            appResult = AnalysisUtils.createResultsObject(
                sha256                      = sha256,
                pkgName                     = pkgName,
                obfuscationTechnique        = obfuscationTechnique,
                status                      = "NO_ANALYZABLE_SMALI_CLASSES",
                numSmaliClasses             = app.numSmaliClasses,
                numSmaliClassesAnalyzed     = 0,
                llmFinalLabel               = None
            )
            continue

        # Determine the final predicted label for the app based on majority voting across analyzed classes, using the label counts. In case of ties, prefer the label of the most recently analyzed class.
        llmFinalLabel = max(EXPECTED_LABELS, key=lambda label: (appLabelCounts.get(label, 0), label == classLabel))
        isCorrect = llmFinalLabel == obfuscationTechnique

        print("\n--- 🎯 Results for App             : {}".format(pkgName))
        print("--- 🔹 Ground Truth Label          : {}".format(obfuscationTechnique))
        print("--- 🔹 Predicted Label             : {}".format(llmFinalLabel))
        print("--- 🔹 Prediction Counts           : {}".format(dict(appLabelCounts)))
        print("--- 🔹 N. Skipped for Context      : {}".format(numSkippedForContextThreshold))
        print("--- 🔹 N. Skipped for LLM Output   : {}".format(numSkippedForInvalidLlmOutput))
        print("--- 🔹 Correct Prediction          : {}".format(isCorrect))

        appResult = AnalysisUtils.createResultsObject(
            sha256                      = sha256,
            pkgName                     = pkgName,
            obfuscationTechnique        = obfuscationTechnique,
            status                      = "SUCCESS",
            numSmaliClasses             = app.numSmaliClasses,
            numSmaliClassesAnalyzed     = effectiveNumSmaliClassesAnalyzed,
            llmFinalLabel               = llmFinalLabel,
            labelFrequency              = dict(appLabelCounts)
        )

    except Exception as exc:
        print("--- ⚠️ Error while analyzing {}: {}".format(pkgName, exc))
        print(traceback.format_exc())
        errorTrace = traceback.format_exc().replace("\n", " | ")
        appResult = AnalysisUtils.createResultsObject(
            sha256                      = sha256,
            pkgName                     = pkgName,
            obfuscationTechnique        = obfuscationTechnique,
            status                      = "ERROR - {}".format(errorTrace),
            numSmaliClasses             = 0 if app is None else app.numSmaliClasses,
            numSmaliClassesAnalyzed     = 0,
            llmFinalLabel               = None
        )

    # Finally, save the results for this app (whether success or error) and clean up the app's decompiled files to save disk space. Also, print a separator before moving on to the next app.
    finally:
        if appResult is not None:
            classificationResults.append(appResult)
            AnalysisUtils.saveResults(classificationResults, OUTPUT_PATH_JSON)
            print("\n--- 💾 Partial Report saved!")

        if app is not None:
            app.deleteAPK()
            app.deleteAll()

        print("\n" + "+++" * 20)


##### 💾 Save Results


In [ ]:
AnalysisUtils.saveResults(classificationResults, OUTPUT_PATH_JSON)
AnalysisUtils.saveResultsAsCsv(classificationResults, OUTPUT_PATH_CSV)
print("--- 💾 JSON Report saved : {}".format(OUTPUT_PATH_JSON))
print("--- 💾 CSV Report saved  : {}".format(OUTPUT_PATH_CSV))

#### 📊 Print some stats/results


In [ ]:
resultsDF = pd.DataFrame(classificationResults)
successDF = resultsDF[resultsDF["status"] == "SUCCESS"].copy() if len(resultsDF) > 0 else pd.DataFrame()

print("\n--- 📊 Dataset Statistics")
print("--- 🔹 Total Apps              : {}".format(len(resultsDF)))
print("--- 🔹 Success                 : {}".format(len(successDF)))
print("--- 🔹 Error                   : {}".format((resultsDF["status"].astype(str).str.startswith("ERROR -")).sum() if len(resultsDF) > 0 else 0))
print("--- 🔹 Zero Smali Classes      : {}".format((resultsDF["status"] == "NO_SMALI_CLASSES").sum() if len(resultsDF) > 0 else 0))
print("--- 🔹 Context Skipped Apps    : {}".format((resultsDF["status"] == "NO_SMALI_CLASSES_WITHIN_CONTEXT_THRESHOLD").sum() if len(resultsDF) > 0 else 0))

if len(successDF) > 0:
    numCorrect = (successDF["llmFinalLabel"] == obfuscationTechnique).sum()
    print("--- 🔹 Exact-Match Accuracy    : {:.2%}".format(numCorrect / len(successDF)))
    print("--- 🔹 Predicted Label Counts  : {}".format(successDF["llmFinalLabel"].value_counts().to_dict()))

successDF.head(10)

##### 🔚 End


In [ ]:
endTime = datetime.datetime.now()
print("\n🔚 --- END:  {} --- 🔚".format(endTime.strftime("%Y-%m-%d %H:%M:%S")))

totalTime = endTime - initTime
hours = totalTime.total_seconds() // 3600
minutes = (totalTime.total_seconds() % 3600) // 60
print("⏱️ --- Time: {:02d} hours and {:02d} minutes [{:02d} seconds] --- ⏱️".format(int(hours), int(minutes), int(totalTime.total_seconds())))